<a href="https://colab.research.google.com/github/lsteffenel/CHPS0804/blob/main/TP4/Sujet_CNN_garbage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **Garbage classification**
---



Ce notebook vous guidera (partiellement) dans le but de classer des images de dechets à l'aide d'un modèle tf.keras.Sequential. Il démontre les concepts suivants :

* Charger un ensemble de données à partir du disque.
* Identification de l'overfitting et application de techniques pour l'atténuer, notamment l'augmentation des données et le dropout.

A ce fin, vous allez travailler sur différentes étapes :

* Examiner et comprendre les données
* Construire un pipeline d'entrée
* Construire le modèle
* Entraîner le modèle
* Tester le modèle
* Améliorer le modèle et répéter le processus


## Télécharger le dataset

Pour télécharger le dataset, utiliser wget pour récupérer le fichier (ce n'est pas énorme, juste 35MB).

In [ ]:
!wget http://urca.lsteffenel.fr/Garbage.tar.gz

Décompresser le fichier et regarder le contenu. VOus trouverez 6 sous-répertoires, chacun pour une classe différente d'ordures.

In [ ]:
!tar xzf Garbage.tar.gz
!ls Garbage\ classification

## Exploration des données

on commence par quelques "import".

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import PIL
import tensorflow as tf

import keras
from keras import layers
from keras.models import Sequential

Ici, nous allons explorer le répertoire et les données. Tout d'abord, combien d'images on a ?



In [ ]:
import pathlib
data_dir = "./Garbage classification"
data_dir = pathlib.Path(data_dir)

In [ ]:
image_count = len(list(data_dir.glob('*/*.jpg')))
print(image_count)

Et ici un aperçu des images des classes.

In [ ]:
glass = list(data_dir.glob('glass/*'))
PIL.Image.open(str(glass[0]))

In [ ]:
cardboard = list(data_dir.glob('cardboard/*'))
PIL.Image.open(str(cardboard[0]))

In [ ]:
metal = list(data_dir.glob('metal/*'))
PIL.Image.open(str(metal[0]))

In [ ]:
paper = list(data_dir.glob('paper/*'))
PIL.Image.open(str(paper[0]))

## Chargement des données

In [ ]:
#Define some parameters for the loader
batch_size = 32
img_height = 180
img_width = 180

Dans le prochain paragraphe nous allons utiliser une fonction "spéciale" de keras qui permet de charger les images lorsque le dataset est divisé par classes (ce qui est notre cas). Il s'agit de la fonction `keras.utils.image_dataset_from_directory()`.

Renseignez-vous sur la [syntaxe de cette fonction](https://keras.io/api/data_loading/image/#imagedatasetfromdirectory-function) puis complétez la cellule suivante avec les instructions ci-dessous :

- Les images on une taille de 512x384 pixels. Afin de simplifier le traitement, redimensionner à 180x180 pixels.

- séparer le dataset en 80% pour le training et 20% pour la validation.

---



In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
  data_dir,
  subset="training", # indique le sous-répertoire utilisé pour le dataset
  seed=123,
  # validation_split
  # image_size
  batch_size=batch_size)

Faites la même chose pour un dataset  ``
val_ds`.

In [ ]:
val_ds = keras.utils.image_dataset_from_directory(
  # votre code ici

  )

Imprimer le nom des classes.

In [ ]:
class_names = train_ds.class_names
print(class_names)

**Visualisation des données**

Les neuf premières images du jeu de données d'entraînement :






In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

**Standardiser les données**

Les valeurs des canaux RGB se situent dans la plage [0, 255]. Ce n'est pas idéal pour un réseau neuronal ; en général, nous devons chercher à rendre vos valeurs d'entrée petites.


Ici, nous allons normaliser les valeurs pour qu'elles soient dans la plage [0, 1] en utilisant `keras.layers.Rescaling()`, appelé juste au début de `Sequential()`.

**Créer le reste du modèle**

En plus de l'entrée `Input()` et de l'entrée `Rescaling()`, vous devez rajouter à Keras Sequential 5 blocs de convolution (`keras.layers.Conv2D`) avec une couche de pooling (`keras.layers.MaxPooling2D`) après chacune des convolutions. Utiliser respectivement 16, 32, 64, 128 et 256 neurones par couche, et des filtres de convolution 3x3, et l'activation ReLU, et `padding=same`.

Enfin, rajoutez une couche entièrement connectée. Elle est composée d'une Flatten, d'une couche Dense avec 128 unités, activée par une fonction d'activation ReLU ('relu'), puis finalement d'une couche Dense avec autant de neurones que de classes, activée par SoftMax.



In [ ]:
num_classes = len(class_names)

model = Sequential([
  layers.Input(shape=(img_height, img_width, 3)),
  layers.Rescaling(1./255),
  # 5 blocs convolution + maxpooling

  # réseau dense pour la classification

])

Nous choisissons l'optimiseur tf.keras.optimizers.Adam et la fonction de perte tf.keras.losses.SparseCategoricalCrossentropy. Pour afficher la précision de l'entrainement et du test pour chaque époque de l'entrainement.

In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

Nous visualisons toutes les couches du réseau à l'aide de la méthode Keras Model.summary().

In [ ]:
model.summary()

Quelle est la taille du modèle, en Mo ?

Nous entrainons le modèle pendant 20 époques avec la méthode Keras Model.fit :

In [ ]:
epochs=20
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

**Visualisons les résultats de l'entraînement**

Créeation des graphiques de la perte et de la précision sur les ensembles des entraînements et du test.


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

Les graphiques montrent que la précision de l'entrainement et celle de la validation sont très éloignées l'une de l'autre, et que le modèle n'a même pas atteint une accuracy de 70 % sur l'ensemble de validation.

Les sections suivantes montrent comment inspecter ce qui n'a pas fonctionné et essayer d'augmenter la performance globale du modèle.

# **Overfitting**
Dans les graphiques ci-dessus, la précision de l'entraînement augmente linéairement au fil du temps, tandis que la précision de la validation stagne entre 60% et 70 % du processus de l'entraînement. De plus, la différence de précision entre l'entraînement et la validation est notable - un signe de **Overfitting**.

Il existe plusieurs façons de lutter contre l'overfitting dans le processus de l'entraînement. On va utiliser l'augmentation des données et l'ajout du dropout à notre modèle.

#**Data augmentation**

 **L'overfitting** se produit généralement lorsque le nombre d'exemples d'apprentissage est faible. L'augmentation des données consiste à générer des données d'apprentissage supplémentaires à partir de vos exemples existants en les augmentant à l'aide de transformations aléatoires qui produisent des images d'apparence crédible. Cela permet d'exposer le modèle à davantage d'aspects des données et de mieux le généraliser.

 Nous allons mettre en œuvre l'augmentation des données à l'aide des couches de preprocessing Keras suivantes : `keras.layers.RandomFlip()`, `keras.layers.RandomRotation()` et `keras.layers.RandomZoom()`. Elles peuvent être incluses dans votre modèle comme d'autres couches, et exécutées sur le GPU.

In [ ]:
data_augmentation = keras.Sequential(
  [
    layers.Input(shape=(img_height, img_width, 3)),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
  ]
)

Visualisons quelques exemples augmentés en appliquant plusieurs fois l'augmentation des données à la même image :

In [ ]:
plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
  for i in range(9):
    augmented_images = data_augmentation(images)
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(augmented_images[0].numpy().astype("uint8"))
    plt.axis("off")

nous allons ajouter l'augmentation des données à notre modèle avant l'entraînement dans l'étape suivante.

##**Dropout**

Une autre technique permettant de réduire l'overfitting consiste à introduire une régularisation de type dropout dans le réseau.

Lorsque nous appliquons le dropout à une couche, il élimine de manière aléatoire (en fixant l'activation à zéro) un certain nombre d'unités de sortie de la couche pendant le processus de l'entrainement. Le dropout prend un nombre fractionnaire comme valeur d'entrée, sous la forme 0,1, 0,2, 0,4, etc. Cela signifie que 10 %, 20 % ou 40 % des unités de sortie sont éliminées de manière aléatoire de la couche appliquée.


# Création du nouveau modèle

Crééons un nouveau réseau neuronal.

Tout d'abord, on "insère" le modèle pour la data augmentation, puis on fait le Rescaling.

Maintenant, reprensez vos blocs Convolution+MaxPooling et rajoutez un Dropout de 20% après chaque bloc.

Enfin, terminez avec les couches denses.

In [ ]:
model = Sequential([
  data_augmentation, # ceci faut la data augmentation qu'on a vu ci-dessus
  layers.Rescaling(1./255),
  ## 5 blocs Convolution, Maxpooling, Dropout
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dense(num_classes, activation='softmax')
])

#Compiler et entrainer le modèle

In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
epochs = 40
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

#Visualisation de la courbe d'entraînement

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

Après avoir appliqué l'augmentation des données et keras.layers.Dropout, il y a moins de overfitting qu'auparavant, et la précision de l'entrainement et de la validation sont plus proches.

### Remarque :
- IMPORTANT : si vous re-exécutez la commande `fit()`, vous allez continuer l'entraînement à partir du dernier point (dans notre cas, vous faites encore 40 epochs).
- si vous entraînez sur un nombre trop important, la décorrelation entre l'accuracy train et validation sera plus grande, une évidence de sûrentraînement.

**Prédiction sur des données nouvelles**

In [ ]:
paper_path = "./Garbage classification/paper/paper103.jpg"

img = keras.utils.load_img(
    paper_path, target_size=(img_height, img_width)
)
img_array = keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0) # Create a batch

predictions = model.predict(img_array)
score = predictions[0]

print(
    "This image most likely belongs to {} with a {:.2f} percent confidence."
    .format(class_names[np.argmax(score)], 100 * np.max(score))
)

plt.imshow(img_array[0].numpy().astype("uint8"))


In [ ]:
glass_path = "./Garbage classification/glass/glass113.jpg"

img = keras.utils.load_img(
    glass_path, target_size=(img_height, img_width)
)
img_array = keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0) # Create a batch


predictions = model.predict(img_array)
score = predictions[0]

print(
    "This image most likely belongs to {} with a {:.2f} percent confidence."
    .format(class_names[np.argmax(score)], 100 * np.max(score))
)

plt.imshow(img_array[0].numpy().astype("uint8"))


In [ ]:
cardboard_path = "./Garbage classification/cardboard/cardboard4.jpg"

img = keras.utils.load_img(
    cardboard_path, target_size=(img_height, img_width)
)
img_array = keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0) # Create a batch

predictions = model.predict(img_array)
score = predictions[0]

print(
    "This image most likely belongs to {} with a {:.2f} percent confidence."
    .format(class_names[np.argmax(score)], 100 * np.max(score))
)

plt.imshow(img_array[0].numpy().astype("uint8"))


In [ ]:
trash_path = "./Garbage classification/trash/trash10.jpg"

img = keras.utils.load_img(
    trash_path, target_size=(img_height, img_width)
)
img_array = keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0) # Create a batch

predictions = model.predict(img_array)
score = predictions[0]

print(
    "This image most likely belongs to {} with a {:.2f} percent confidence."
    .format(class_names[np.argmax(score)], 100 * np.max(score))
)

plt.imshow(img_array[0].numpy().astype("uint8"))


In [ ]:
plastic_path = "./Garbage classification/plastic/plastic14.jpg"

img = keras.utils.load_img(
    plastic_path, target_size=(img_height, img_width)
)
img_array = keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0) # Create a batch

predictions = model.predict(img_array)
score = predictions[0]

print(
    "This image most likely belongs to {} with a {:.2f} percent confidence."
    .format(class_names[np.argmax(score)], 100 * np.max(score))
)

plt.imshow(img_array[0].numpy().astype("uint8"))


Cela n'empêche que parfois le modèle se trompe, comme ici :

In [ ]:
plastic_path = "./Garbage classification/plastic/plastic10.jpg"

img = keras.utils.load_img(
    plastic_path, target_size=(img_height, img_width)
)
img_array = keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0) # Create a batch

predictions = model.predict(img_array)
score = predictions[0]

print(
    "This image most likely belongs to {} with a {:.2f} percent confidence."
    .format(class_names[np.argmax(score)], 100 * np.max(score))
)

plt.imshow(img_array[0].numpy().astype("uint8"))


Nous remarquons que notre modéle donnes des bonnes résultats dans la préduction sur des données.

# À vous :    
- lors de l'entraînement, l'accuracy subit parfois des hauts et des bas. Utilisez des `callbacks` pour enregistrer le meilleur modèle, puis chargez-le avant de faire l'inférence.

- comme vous savez déjà utiliser des callbacks, rajoutez également un callback pour faire du *early stop* afin que l'entraînement s'arrête au bout de 5 epochs sans amélioration.